**Ce script PySpark sert à simuler et analyser les résultats de 24 matchs de football de la Ligue des Champions**

**Projet Databricks : Pipeline d’Analyse 360° de la Ligue des Champions**

- Rôle cible : Data Analyst Senior / Analytics Engineer 
- Architecture : Médaillon (Bronze ➜ Silver ➜ Gold) 
- Moteur : PySpark / Delta Lake (Compatible Unity Catalog / Spark Connect) 

**1. Creation de données (df_matches) et calcule de df_clean : schéma explicite, données de test et calcul des points à domicile et à l'extérieur.**

In [0]:
# 1. Importation des fonctions de Databricks / PySpark
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Définition de la structure des données
schema = StructType([
    StructField("match_id", IntegerType(), True),
    StructField("home_team", StringType(), True),
    StructField("away_team", StringType(), True),
    StructField("home_goals", IntegerType(), True),
    StructField("away_goals", IntegerType(), True)
])

# 2. Simulation de 24 matchs impliquant de nombreuses équipes européennes
donnees_champions_league = [
    (1, "Paris SG", "Real Madrid", 3, 1),
    (2, "Bayern Munich", "Arsenal", 2, 2),
    (3, "FC Barcelone", "Monaco", 1, 2),
    (4, "Manchester City", "Juventus", 4, 0),
    (5, "Inter Milan", "Liverpool", 0, 2),
    (6, "Bayer Leverkusen", "AC Milan", 2, 1),
    (7, "Atletico Madrid", "RB Leipzig", 1, 0),
    (8, "Borussia Dortmund", "Sporting CP", 3, 2),
    (9, "Benfica", "Feyenoord", 1, 1),
    (10, "PSV Eindhoven", "Girona", 2, 0),
    (11, "Aston Villa", "Bologna", 2, 1),
    (12, "Celtic Glasgow", "Club Brugge", 0, 0),
    (13, "Stuttgart", "Lille OSC", 1, 2),
    (14, "Eintracht Frankfurt", "Lazio", 2, 3),
    (15, "Real Sociedad", "AS Roma", 1, 1),
    (16, "Shakhtar Donetsk", "Dinamo Zagreb", 0, 2),
    (17, "Salzburg", "Brest", 1, 3),
    (18, "Sparta Prague", "Young Boys", 2, 0),
    (19, "Sturm Graz", "Sporting CP", 0, 2),
    (20, "Crvena Zvezda", "Monaco", 1, 4),
    (21, "Real Madrid", "Borussia Dortmund", 5, 2),
    (22, "Paris SG", "PSV Eindhoven", 1, 1),
    (23, "Juventus", "VfB Stuttgart", 0, 1),
    (24, "Arsenal", "Shakhtar Donetsk", 1, 0)
]

# 3. Création du DataFrame global
df_matches = spark.createDataFrame(donnees_champions_league, schema)

# 4. Calcul des points à domicile et à l'extérieur
df_clean = df_matches.withColumn(
    "home_points", 
    F.when(F.col("home_goals") > F.col("away_goals"), 3)
     .when(F.col("home_goals") == F.col("away_goals"), 1)
     .otherwise(0)
).withColumn(
    "away_points",
    F.when(F.col("away_goals") > F.col("home_goals"), 3)
     .when(F.col("away_goals") == F.col("home_goals"), 1)
     .otherwise(0)
)

print("Nouveau jeu de données de la Ligue des Champions chargé avec succès !")



**2. Agrégation et classement : Ce script PySpark génère le classement général officiel à partir des résultats des matchs précédents**

La fusion des résultats domicile/extérieur et l'agrégation par équipe sont correctes + un critère de départage pour obtenir un ordre totalement déterministe en cas d'égalité

In [0]:

# ============================================================
# CELLULE 2 — PRÉPARATION DES DONNÉES ET CLASSEMENT GÉNÉRAL
# Projet : Champions League Analytics
# Technologies : Databricks / PySpark
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

# 1. Définition du schéma
schema = StructType([
    StructField("match_id", IntegerType(), False),
    StructField("home_team", StringType(), False),
    StructField("away_team", StringType(), False),
    StructField("home_goals", IntegerType(), False),
    StructField("away_goals", IntegerType(), False)
])

# 2. Chargement du jeu de données simulé
donnees_champions_league = [
    (1, "Paris SG", "Real Madrid", 3, 1),
    (2, "Bayern Munich", "Arsenal", 2, 2),
    (3, "FC Barcelone", "Monaco", 1, 2),
    (4, "Manchester City", "Juventus", 4, 0),
    (5, "Inter Milan", "Liverpool", 0, 2),
    (6, "Bayer Leverkusen", "AC Milan", 2, 1),
    (7, "Atletico Madrid", "RB Leipzig", 1, 0),
    (8, "Borussia Dortmund", "Sporting CP", 3, 2),
    (9, "Benfica", "Feyenoord", 1, 1),
    (10, "PSV Eindhoven", "Girona", 2, 0),
    (11, "Aston Villa", "Bologna", 2, 1),
    (12, "Celtic Glasgow", "Club Brugge", 0, 0),
    (13, "Stuttgart", "Lille OSC", 1, 2),
    (14, "Eintracht Frankfurt", "Lazio", 2, 3),
    (15, "Real Sociedad", "AS Roma", 1, 1),
    (16, "Shakhtar Donetsk", "Dinamo Zagreb", 0, 2),
    (17, "Salzburg", "Brest", 1, 3),
    (18, "Sparta Prague", "Young Boys", 2, 0),
    (19, "Sturm Graz", "Sporting CP", 0, 2),
    (20, "Crvena Zvezda", "Monaco", 1, 4),
    (21, "Real Madrid", "Borussia Dortmund", 5, 2),
    (22, "Paris SG", "PSV Eindhoven", 1, 1),
    (23, "Juventus", "VfB Stuttgart", 0, 1),
    (24, "Arsenal", "Shakhtar Donetsk", 1, 0)
]

df_matches = spark.createDataFrame(
    donnees_champions_league,
    schema
)

# 3. Calcul des points à domicile et à l'extérieur
df_clean = (
    df_matches
    .withColumn(
        "home_points",
        F.when(F.col("home_goals") > F.col("away_goals"), 3)
         .when(F.col("home_goals") == F.col("away_goals"), 1)
         .otherwise(0)
    )
    .withColumn(
        "away_points",
        F.when(F.col("away_goals") > F.col("home_goals"), 3)
         .when(F.col("away_goals") == F.col("home_goals"), 1)
         .otherwise(0)
    )
)

# 4. Extraction des statistiques à domicile
df_home = df_clean.select(
    F.col("home_team").alias("team"),
    F.col("home_points").alias("points"),
    F.col("home_goals").alias("goals_scored"),
    F.col("away_goals").alias("goals_conceded")
)

# 5. Extraction des statistiques à l'extérieur
df_away = df_clean.select(
    F.col("away_team").alias("team"),
    F.col("away_points").alias("points"),
    F.col("away_goals").alias("goals_scored"),
    F.col("home_goals").alias("goals_conceded")
)

# 6. Consolidation des statistiques
df_all_matches = df_home.unionByName(df_away)

# 7. Agrégation des performances par équipe
df_ranking = df_all_matches.groupBy("team").agg(
    F.count("*").alias("matches_played"),
    F.sum("points").alias("total_points"),
    F.sum("goals_scored").alias("total_scored"),
    F.sum("goals_conceded").alias("total_conceded")
)

# 8. Calcul de la différence de buts et classement
df_final_ladder = (
    df_ranking
    .withColumn(
        "goal_difference",
        F.col("total_scored") - F.col("total_conceded")
    )
    .orderBy(
        F.col("total_points").desc(),
        F.col("goal_difference").desc(),
        F.col("total_scored").desc(),
        F.col("team").asc()
    )
)

# 9. Affichage du classement
print("Classement général calculé avec succès.")

display(df_final_ladder)


**3. Calcul du rang et du statut UEFA :** 

Applique les règles de qualification officielles du nouveau format de la Ligue des Champions au tableau précédemment trié - Identifier les Qualifiés, Barragistes et Éliminés 

Le code suivant conserve tes trois critères sportifs et ajoute le nom de l'équipe comme critère de stabilité du classement.

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_spec = Window.orderBy(
    F.col("total_points").desc(),
    F.col("goal_difference").desc(),
    F.col("total_scored").desc(),
    F.col("team").asc()
)

df_uefa_status = (
    df_final_ladder
    .withColumn("rank", F.row_number().over(window_spec))
    .withColumn(
        "uefa_status",
        F.when(F.col("rank") <= 8, "Qualifié direct")
         .when(F.col("rank") <= 24, "Barragiste")
         .otherwise("Éliminé")
    )
)

display(
    df_uefa_status.select(
        "rank",
        "team",
        "matches_played",
        "total_points",
        "goal_difference",
        "total_scored",
        "uefa_status"
    )
)


Databricks visualization. Run in Databricks to view.

**4. Analyse défensive**
\[
\text{Buts encaissés par match}
=\frac{\text{Total des buts encaissés}}{\text{Nombre de matchs}}
\]

PySpark pour l'analyse défensive - Isole et analyse les performances défensives de chaque club de la compétition afin d'identifier les équipes les plus solides et les plus vulnérables. 

In [0]:

from pyspark.sql import functions as F

df_defense = (
    df_final_ladder
    .withColumn(
        "goals_conceded_per_match",
        F.round(
            F.col("total_conceded") / F.col("matches_played"),
            2
        )
    )
    .orderBy(
        F.col("goals_conceded_per_match").asc(),
        F.col("goal_difference").desc(),
        F.col("team").asc()
    )
)

df_defense_display = df_defense.select(
    F.col("team").alias("Team"),
    F.col("matches_played").alias("Matches_Played"),
    F.col("total_conceded").alias("Goals_Conceded"),
    F.col("goals_conceded_per_match").alias(
        "Goals_Conceded_Per_Match"
    )
)

display(df_defense_display)


Databricks visualization. Run in Databricks to view.

**5. Générer et enregistrer les graphiques**

In [0]:

import os
import matplotlib.pyplot as plt

# Conversion des résultats agrégés uniquement
df_plot = df_final_ladder.select(
    "team",
    "total_points",
    "goal_difference",
    "total_scored",
    "total_conceded",
    "matches_played"
).toPandas()

df_defense_plot = df_defense.select(
    "team",
    "goals_conceded_per_match"
).toPandas()

# Dossier de sortie
os.makedirs("images", exist_ok=True)

# --------------------------------------------------
# GRAPHIQUE 1 : Classement par nombre de points
# --------------------------------------------------
df_plot = df_plot.sort_values(
    ["total_points", "goal_difference", "total_scored", "team"],
    ascending=[True, True, True, False]
)

fig, ax = plt.subplots(figsize=(11, 8))

bars = ax.barh(
    df_plot["team"],
    df_plot["total_points"]
)

ax.bar_label(bars, padding=3, fontsize=9)
ax.set_title("Champions League — Points par équipe")
ax.set_xlabel("Total des points")
ax.set_ylabel("Équipe")
ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(
    "images/champions_league_ranking.png",
    dpi=160,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()
plt.close(fig)

# --------------------------------------------------
# GRAPHIQUE 2 : Performance défensive
# --------------------------------------------------
df_defense_plot = df_defense_plot.sort_values(
    ["goals_conceded_per_match", "team"],
    ascending=[False, False]
)

fig, ax = plt.subplots(figsize=(11, 8))

bars = ax.barh(
    df_defense_plot["team"],
    df_defense_plot["goals_conceded_per_match"]
)

ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
ax.set_title("Champions League — Buts encaissés par match")
ax.set_xlabel("Buts encaissés par match")
ax.set_ylabel("Équipe")
ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(
    "images/champions_league_defense.png",
    dpi=160,
    bbox_inches="tight",
    facecolor="white"
)
plt.show()
plt.close(fig)

print("Graphiques enregistrés dans le dossier images/")
